# OSL RSAC Notebook — Bilateral Sensor + Larva Connectome

Recurrent SAC on the same biological env as the PPO notebook. Single-env episode loop (off-policy), with selectable actor backbone for ablation:
- `connectome` — larva connectome + efference concat (default)
- `gru` — GRU-only baseline
- `mlp` — feed-forward baseline

Critic: GRU-based twin Q-net (`QCritic` × 2). Pure tanh-Gaussian action `[v, body_omega, head_omega]`.

Repo clone is mandatory for the connectome backbone (CSVs in `assets/connectome/`).

In [ ]:
# Colab setup. Re-running is safe.
import os, sys, subprocess

REPO_URL = 'https://github.com/InHyunseo/Brain-inspired-OSL.git'
REPO_DIR = '/content/2d-osl' if os.path.isdir('/content') else os.path.abspath('2d-osl')
if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone', REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

%pip install --quiet -r requirements.txt

for p in ('assets/connectome/weights.csv', 'assets/connectome/metadata.csv'):
    assert os.path.exists(p), f'Missing {p} — clone failed?'
print('repo:', REPO_DIR, '\ncwd :', os.getcwd())

## Smoke check

In [ ]:
import torch, numpy as np
from src.envs.osl_env import OslEnv
from src.agents.rsac_agent import RSACAgent

env = OslEnv()
print('obs', env.observation_space.shape, 'action', env.action_space.shape)
for backbone in ('connectome', 'gru', 'mlp'):
    agent = RSACAgent(
        obs_dim=env.observation_space.shape[0], act_dim=env.action_space.shape[0],
        action_low=env.action_space.low, action_high=env.action_space.high,
        device=torch.device('cpu'), actor_backbone=backbone,
    )
    a, h = agent.get_action(np.zeros(5, dtype=np.float32))
    print(f'{backbone:10s} action={a}  hidden={(h.shape if h is not None else None)}')

## Training
Single env, off-policy episode loop. RSAC has no explicit curriculum hook; instead pick a fixed noise stage (set `NOISE_STAGE` / `NOISE_STRENGTH`). Bump `TOTAL_EPISODES` for serious runs.

### Live TensorBoard

In [ ]:
# ===== TensorBoard (run BEFORE the training cell to watch curves live) =====
# Works in Colab and local Jupyter. Writers go to <RUN_DIR>/tb/ inside the run.
import os
os.makedirs('runs', exist_ok=True)
%load_ext tensorboard
%tensorboard --logdir runs --port 6006


In [ ]:
# ===== HYPERPARAMETERS — edit freely =====
ACTOR_BACKBONE = 'connectome'   # 'connectome' | 'gru' | 'mlp'
TOTAL_EPISODES = 1000
LEARNING_STARTS = 200
BATCH_SIZE = 128
SEQ_LEN = 16
BUFFER_CAP_STEPS = 150_000
LOG_EVERY = 20

# Env (same as PPO notebook)
ENV_KW = dict(
    sensor_spacing_mm=0.15, episode_seconds=60.0,
    arena_width_mm=80.0, arena_height_mm=120.0,
    source_x_mm=40.0, source_y_mm=100.0,
    gaussian_sigma_mm=30.0, success_radius_mm=7.5,
)
NOISE_STAGE = 0
NOISE_STRENGTH = 0.0

# Agent
AGENT_KW = dict(
    rnn_hidden=147,
    lr_actor=3e-4, lr_critic=3e-4, lr_alpha=3e-4,
    gamma=0.99, tau=0.005,
    connectome_latent_dim=32,
    connectome_message_passing_steps=6,
    connectome_weights_csv='assets/connectome/weights.csv',
    connectome_metadata_csv='assets/connectome/metadata.csv',
)
SEED = 0
# ==========================================

import os, time, json
import torch, numpy as np
from collections import deque
from src.envs.osl_env import EnvConfig, OslEnv
from src.agents.rsac_agent import RSACAgent
from src.utils.buffer import EpisodeReplayBuffer

RUN_DIR = os.path.join('runs', f'rsac_nb_{ACTOR_BACKBONE}_{time.strftime("%Y%m%d_%H%M%S")}')
os.makedirs(os.path.join(RUN_DIR, 'checkpoints'), exist_ok=True)
os.makedirs(os.path.join(RUN_DIR, 'plots'), exist_ok=True)
with open(os.path.join(RUN_DIR, 'config.json'), 'w') as f:
    json.dump({'env': ENV_KW, 'agent': AGENT_KW, 'noise': (NOISE_STAGE, NOISE_STRENGTH),
               'episodes': TOTAL_EPISODES, 'backbone': ACTOR_BACKBONE, 'seed': SEED}, f, indent=2)
print('[run_dir]', RUN_DIR)

# TensorBoard writer (mirrors per-LOG_EVERY scalars to <RUN_DIR>/tb)
try:
    from torch.utils.tensorboard import SummaryWriter
    tb_writer = SummaryWriter(log_dir=os.path.join(RUN_DIR, 'tb'))
except Exception as exc:
    print(f'[tb] disabled ({exc})')
    tb_writer = None

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

env_cfg = {**ENV_KW, 'noise_stage': NOISE_STAGE, 'noise_strength': NOISE_STRENGTH, 'seed': SEED}
env = OslEnv(EnvConfig.from_dict(env_cfg))
agent = RSACAgent(
    obs_dim=env.observation_space.shape[0], act_dim=env.action_space.shape[0],
    action_low=env.action_space.low, action_high=env.action_space.high,
    device=device, actor_backbone=ACTOR_BACKBONE, **AGENT_KW,
)
buffer = EpisodeReplayBuffer(cap_steps=BUFFER_CAP_STEPS)

ep_returns, ep_steps_to_goal = [], []
best_return = -float('inf')

for ep in range(1, TOTAL_EPISODES + 1):
    obs, _ = env.reset(seed=SEED + ep)
    h = None
    traj, ep_ret, steps_in_ep, success_step = [], 0.0, 0, None
    while True:
        action, h = agent.get_action(obs, h)
        next_obs, r, term, trunc, info = env.step(action)
        traj.append((obs, action, float(r), next_obs, float(term)))
        obs = next_obs; ep_ret += float(r); steps_in_ep += 1
        if info.get('success') and success_step is None:
            success_step = steps_in_ep
        if term or trunc:
            break
    buffer.add_episode(traj)
    ep_returns.append(ep_ret)
    ep_steps_to_goal.append(success_step if success_step is not None else steps_in_ep)

    if len(buffer) >= LEARNING_STARTS:
        agent.update(buffer.sample(BATCH_SIZE, SEQ_LEN))

    if ep_ret > best_return:
        best_return = ep_ret
        agent.save(os.path.join(RUN_DIR, 'checkpoints', 'best.pt'))

    if ep % LOG_EVERY == 0:
        recent = ep_returns[-LOG_EVERY:]
        print(f'[ep {ep:5d}] return={ep_ret:7.2f} recent_mean={np.mean(recent):7.2f} steps={steps_in_ep} buffer={len(buffer)}')
        if tb_writer is not None:
            tb_writer.add_scalar('return/episode', float(ep_ret), ep)
            tb_writer.add_scalar('return/recent_mean', float(np.mean(recent)), ep)
            tb_writer.add_scalar('steps/episode', int(steps_in_ep), ep)
            tb_writer.add_scalar('buffer/size', int(len(buffer)), ep)
            tb_writer.add_scalar('eval/best_return', float(best_return), ep)
            tb_writer.flush()

agent.save(os.path.join(RUN_DIR, 'checkpoints', 'final.pt'))
print('[done] best_return:', round(best_return, 2))

if tb_writer is not None:
    tb_writer.close()


## Training curve

In [ ]:
import matplotlib.pyplot as plt

def ema(arr, alpha=0.05):
    out, m = [], arr[0] if arr else 0.0
    for x in arr:
        m = alpha * x + (1 - alpha) * m
        out.append(m)
    return out

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
x = list(range(1, len(ep_returns) + 1))
ax[0].plot(x, ep_returns, alpha=0.25, label='raw')
ax[0].plot(x, ema(ep_returns), linewidth=2, label='ema')
ax[0].set_xlabel('episode'); ax[0].set_ylabel('return'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(x, ep_steps_to_goal, alpha=0.4)
ax[1].set_xlabel('episode'); ax[1].set_ylabel('steps to source / episode length'); ax[1].grid(alpha=.3)
fig.tight_layout(); plt.show()

## Evaluation + GIF
Loads `best.pt` (highest training return). Star markers = `event_is_high_cast_like` events.

In [ ]:
# ===== EVAL HYPERPARAMETERS =====
EVAL_NOISE_STAGE = 1
EVAL_NOISE_STRENGTH = 0.5
EVAL_EPISODES = 20
EVAL_SEED_BASE = 20_000
EVAL_CKPT = 'best.pt'  # or 'final.pt'
# ================================

import os, torch, numpy as np
from IPython.display import Image as DisplayImage, display
from src.envs.osl_env import EnvConfig, OslEnv
from src.agents.rsac_agent import RSACAgent
from src.utils.plotter import render_rollout_frame, save_gif

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def make_eval_env(seed):
    cfg_dict = {**ENV_KW, 'noise_stage': EVAL_NOISE_STAGE, 'noise_strength': EVAL_NOISE_STRENGTH, 'seed': seed}
    return OslEnv(EnvConfig.from_dict(cfg_dict))

env = make_eval_env(EVAL_SEED_BASE)
agent = RSACAgent(
    obs_dim=env.observation_space.shape[0], act_dim=env.action_space.shape[0],
    action_low=env.action_space.low, action_high=env.action_space.high,
    device=device, actor_backbone=ACTOR_BACKBONE, **AGENT_KW,
)
agent.load(os.path.join(RUN_DIR, 'checkpoints', EVAL_CKPT))

rets, succ, best = [], [], (-float('inf'), None)
for i in range(EVAL_EPISODES):
    seed = EVAL_SEED_BASE + i
    env = make_eval_env(seed)
    obs, _ = env.reset(seed=seed)
    h = None; ret = 0.0; success = False
    while True:
        action, h = agent.get_action_deterministic(obs, h)
        obs, r, term, trunc, info = env.step(action)
        ret += float(r)
        if info.get('success'):
            success = True
        if term or trunc: break
    rets.append(ret); succ.append(float(success))
    if ret > best[0]: best = (ret, seed)
print(f'success_rate={np.mean(succ):.3f} avg_return={np.mean(rets):.2f} best_seed={best[1]} best_return={best[0]:.2f}')

# Render best episode
best_seed = best[1]
env = make_eval_env(best_seed)
obs, _ = env.reset(seed=best_seed)
h = None
frames, traj_x, traj_y, cast_x, cast_y = [], [], [], [], []
for t in range(env.max_steps):
    action, h = agent.get_action_deterministic(obs, h)
    traj_x.append(env.x_mm); traj_y.append(env.y_mm)
    obs, _, term, trunc, info = env.step(action)
    if info.get('event_is_high_cast_like'):
        cast_x.append(env.x_mm); cast_y.append(env.y_mm)
    frames.append(render_rollout_frame(env, traj_x, traj_y, cast_x, cast_y, t,
                                       title=f'RSAC ({ACTOR_BACKBONE}) seed={best_seed} step={t}'))
    if term or trunc: break

gif_path = os.path.join(RUN_DIR, 'plots', 'best_agent.gif')
save_gif(frames, gif_path, fps=15)
display(DisplayImage(data=open(gif_path, 'rb').read(), format='gif'))